In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_4_day_1.csv',
    'prices_round_4_day_2.csv',
    'prices_round_4_day_3.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import linregress

# --- 1. CONFIGURATION ---
JUMP_THRESHOLD_SD = 1.5   # Sensitivity for the 'Step Jump'
CONFIRM_TICKS = 1e4       # Ticks outside bounds to trigger snap
PULLBACK_STRENGTH = 2e-8 # Constant gravity toward black line
SNAP_RATIO = 0.9        # Snap to 75% of the move to avoid overshooting spikes
VOL_WINDOW = 300

# --- 2. THE DAMPED PLATEAU FILTER ---
def apply_damped_plateau_filter(series, slope, intercept, jump_sd, confirm_period, pullback, snap_ratio):
    local_means = []
    current_local_mean = series.iloc[0]
    outside_counter = 0
    
    # Pre-calculate rolling volatility
    rolling_vol = series.diff().rolling(VOL_WINDOW).std().ffill().bfill().clip(lower=1.0)
    
    for i, val in enumerate(series):
        # A) Global Anchor Projection
        global_target = intercept + (slope * i)
        
        # B) Continuous Reversion (The "Slow Pull")
        current_local_mean += slope # Respect the global trend drift
        current_local_mean -= pullback * (current_local_mean - global_target)
        
        # C) Step Jump Logic with Damping
        sigma = rolling_vol.iloc[i]
        upper_limit = current_local_mean + (jump_sd * sigma)
        lower_limit = current_local_mean - (jump_sd * sigma)
        
        if val > upper_limit or val < lower_limit:
            outside_counter += 1
        else:
            outside_counter = 0
            
        if outside_counter >= confirm_period:
            # DAMPED SNAP: Move most of the way, but not all the way.
            # current_local_mean = (Old Mean * 25%) + (New Price * 75%)
            current_local_mean = (current_local_mean * (1 - snap_ratio)) + (val * snap_ratio)
            outside_counter = 0
            
        local_means.append(current_local_mean)
        
    return pd.Series(local_means, index=series.index)

# --- 3. DATA & EXECUTION ---
df_full = df_total[df_total['product'] == "HYDROGEL_PACK"].copy().sort_values(['day', 'timestamp'])
df_full['global_tick'] = range(len(df_full))
df_full['mid_price'] = df_full['mid_price'].replace(0, np.nan).ffill()

# Global slope calculation
slope, intercept, _, _, _ = linregress(df_full['global_tick'], df_full['mid_price'])
slope = 0

df_full['local_mean'] = apply_damped_plateau_filter(
    df_full['mid_price'], slope, intercept, 
    JUMP_THRESHOLD_SD, CONFIRM_TICKS, PULLBACK_STRENGTH, SNAP_RATIO
)

# Z-Score Calculation
df_full['err'] = df_full['mid_price'] - df_full['local_mean']
df_full['rolling_std'] = df_full['err'].rolling(500).std().clip(lower=1.0)
df_full['z_score'] = df_full['err'] / df_full['rolling_std']

# --- 4. VISUALIZATION ---
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])

# Subplot 1: Price and Means
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=df_full['mid_price'], 
                         line=dict(color='lightgrey', width=1), name='Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=df_full['local_mean'], 
                         line=dict(color='red', width=2), name='Damped Plateau Mean'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=intercept + slope*df_full['global_tick'], 
                         line=dict(color='black', dash='dot'), name='Global Mean'), row=1, col=1)

# Subplot 2: Signal
fig.add_trace(go.Scatter(x=df_full['global_tick'], y=df_full['z_score'], 
                         line=dict(color='blue'), name='Z-Score'), row=2, col=1)
fig.add_hline(y=2.0, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-2.0, line_dash="dash", line_color="red", row=2, col=1)

fig.update_layout(height=800, title="Damped Plateau: Partial Snaps + Constant Reversion", template="plotly_white")
fig.show()